# Prática · Ensinar uma máquina a prever a água turva

**85 minutos, em pares** · Workshop 2 · Projecto EYES TO EARTH

**A missão:** construir, do princípio ao fim, um modelo que prevê se a água do rio Incomáti vai estar **turva** (barrenta).
É o mesmo tipo de modelo que o projecto usa: um **Gradient Boosting**.

## Antes de começar: o que vamos fazer (5 min)

Leiam esta parte com calma, em pares.

### O problema

O rio **Incomáti** atravessa o distrito de Marracuene e chega ao mar, na baía de Maputo. A cor da sua água muda de dia para
dia: umas vezes está mais limpa, outras está **turva** (barrenta), cheia de terra fina. Para quem usa o rio, para regar,
pescar ou buscar água, é útil saber como vai estar a água.

O satélite consegue ver a cor do rio, mas só temos imagens limpas em cerca de **90 manhãs por ano**. Nos outros dias, o
satélite não passou ou havia nuvens.

### A pergunta de hoje

> **Será que um computador consegue aprender, com exemplos do passado, a prever se a água do rio vai estar turva?**

### O que vão fazer

Vão ser **vocês** a ensinar o computador, em cinco etapas:

1. **Conhecer os dados** (Passos 0 a 2): abrir a tabela e perceber o que ela mostra.
2. **Preparar o exame** (Passos 3 a 5): escolher as pistas, separar o treino do teste e encontrar a nota a bater.
3. **Treinar** (Passos 6 e 7): ensinar três algoritmos e ver quanto acertam.
4. **Descobrir o que faltava** (Passo 8): encontrar a pista que faz o modelo funcionar.
5. **Usar o modelo** (Passos 9 a 11): ver os erros, prever uma manhã nova e juntar tudo num só script.

**No fim**, terão um modelo treinado por vocês e saberão dizer quanto ele acerta e onde erra. Não é preciso saber
programar: cada passo traz uma pista, e o computador diz se está certo.

## Os dados que vamos usar

Tudo está numa só tabela, o ficheiro `cor_do_rio_e_mare.csv`:

- **2 205 linhas**. Cada linha é **uma manhã num troço do rio**.
- **778 manhãs** diferentes, de **06/01/2017 a 26/09/2026**: quase 10 anos.
- **3 troços do rio** em Marracuene: o **troço médio** (o mais longe do mar), **jusante** e **foz** (onde o rio chega ao mar).

### Quantas manhãs há em cada ano

| Ano | 2017 | 2018 | 2019 | 2020 | 2021 | 2022 | 2023 | 2024 | 2025 | 2026* |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Manhãs** | 43 | 25 | 86 | 91 | 91 | 89 | 89 | 90 | 93 | 81 |
| **Serve para** | treino | treino | treino | treino | treino | treino | teste | teste | teste | teste |

\* Até 26/09/2026. Em 2017 e 2018 há menos manhãs porque havia menos imagens desta zona.

### De onde vem cada informação

| Informação | Coluna | De onde vem |
| :--- | :--- | :--- |
| **A cor da água** | `ndti` | **Medida pelo satélite Sentinel-2** (programa europeu Copernicus), por volta das 9h30. Compara a luz vermelha com a verde: quanto mais alto, mais barrenta |
| **A água que passa no rio** | `caudal_m3s` | Modelo **GloFAS** (Copernicus), para Marracuene. É **calculada** por computador, não medida no rio |
| **A chuva** | `chuva_7d_mm` | **ERA5** (centro europeu ECMWF): a chuva média em toda a bacia do Incomáti, somada nos 7 dias que acabam nesse dia |
| **A maré** | `altura_da_mare_m` | **Calculada** pelo projecto para a baía de Maputo, à hora do satélite. 0 é o nível médio do mar |
| **A resposta certa** | `turva` | **Calculada por nós** a partir do `ndti`: 1 se a água estava mais barrenta do que o normal nesse troço (a mediana de 2017 a 2022); 0 se estava mais limpa |

### Como se lê uma linha

No dia **24/01/2026**, no meio da grande cheia, na **foz**: passavam **1 701 m³/s** de água,
tinham caído **56 mm** de chuva em 7 dias, a maré estava a **0,44 m** e a água
estava **turva** (`turva` = 1).

### Um cuidado importante

Estes dados mostram a **cor** da água. **Não dizem se a água é segura para beber:** o satélite não vê bactérias nem produtos
químicos, e ninguém recolheu amostras no rio para esta tabela.

## O plano da aula

| Passo | O que vão fazer | Minutos |
| :--- | :--- | :--- |
| Início | Ler o que vamos fazer e conhecer os dados | 5 |
| 0 | Preparar o computador | 5 |
| 1 | Abrir os dados | 5 |
| 2 | Olhar para os dados | 5 |
| 3 | Escolher as pistas e o rótulo | 7 |
| 4 | Separar o treino e o teste | 8 |
| 5 | Encontrar a nota a bater | 3 |
| 6 | Treinar uma árvore de decisão | 10 |
| 7 | Treinar uma floresta e um Gradient Boosting | 10 |
| 8 | Juntar a pista que faltava: a maré | 10 |
| 9 | Ver os erros do modelo | 7 |
| 10 | Prever uma manhã nova | 5 |
| 11 | O script inteiro | 5 |

## Como trabalhar

1. Cliquem numa célula de código (as cinzentas).
2. **Tirem o `#`** do início das linhas e **troquem os `...`** pela resposta.
3. Carreguem em **Shift + Enter**.
4. Leiam a resposta: **«Certo!»**, ou **«Ainda não»** com uma pista.

Corram as células **por ordem**, de cima para baixo. Guardem com **Ctrl + S**.

| Se a última linha do erro disser | O que fazer |
| :--- | :--- |
| `ellipsis`, `Ellipsis` ou `'...'` | Ainda há `...` por trocar |
| `NameError` | Saltaram uma célula: corram as células por ordem |
| `SyntaxError` | Falta uma aspa ou um parêntese, ou ficou mal tirado um `#` |
| `KeyError` | O nome de uma coluna está mal escrito |
| `FileNotFoundError` | O nome do ficheiro está mal escrito |
| `ValueError` | Escreveram um número com vírgula? Em Python as décimas têm ponto: `0.7` |
| `NotFittedError` | Falta treinar o modelo: `.fit(...)` |

Se nada resultar: menu **Kernel**, **Restart Kernel and Run All Cells**.

## Passo 0 · Preparar o computador (5 min)

**O que vamos fazer agora:** confirmar que o computador tem tudo o que a aula precisa.  
**Porquê:** Assim, nenhum problema técnico nos interrompe a meio.

Corram a célula de baixo sem mudar nada. Tem de aparecer **PRONTO**.

In [ ]:
# PREPARAR A AULA: corram esta célula sem mudar nada (Shift + Enter).
import base64, json, os, sys, warnings
from IPython.display import display
warnings.filterwarnings("ignore")

FICHEIRO = "cor_do_rio_e_mare.csv"
_pronto = True
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.metrics import accuracy_score
except ImportError as _erro:
    _pronto = False
    print("FALTA A BIBLIOTECA " + str(_erro.name) + " neste Python: " + sys.executable)
    print()
    print("COMO RESOLVER (escolham uma):")
    print("  1. Menu Kernel > Change Kernel, e escolham o Python do workshop: «Python (Workshop 2 - jupyter_env)».")
    print("  2. Fechem este Jupyter e abram-no pelo Anaconda (Anaconda Navigator > Jupyter Notebook).")
    print("Depois corram esta célula outra vez. Se não resultar, chamem um facilitador.")
    print()
if not os.path.exists(FICHEIRO):
    _pronto = False
    print("FALTA O FICHEIRO " + FICHEIRO + ". Abram este notebook na pasta onde ele está, ao lado dos dados.")

_SEM_MARE = ["caudal_m3s", "chuva_7d_mm", "mes"]
_COM_MARE = _SEM_MARE + ["altura_da_mare_m"]
_RESPOSTAS = json.loads(base64.b64decode("eyJQMSI6IFsiYiIsICJPIG1vZGVsbyBqw6Egdml1IG9zIGV4ZW1wbG9zIGRvIHRyZWlubzogYSBub3RhIHNhaXJpYSBib2EgZGUgbWFpcyBlIG7Do28gcHJvdmFyaWEgbmFkYS4iLCAiSW1hZ2luZW0gdW0gZXhhbWUgY29tIGFzIG1lc21hcyBwZXJndW50YXMgcXVlIGrDoSB2aXJhbSByZXNvbHZpZGFzLiJdLCAiUDIiOiBbImIiLCAiQ29tIGEgbWFyw6ksIHRvZG9zIG9zIGFsZ29yaXRtb3Mgc29iZW0uIEEgcGlzdGEgY2VydGEgdmFsZSBtYWlzIGRvIHF1ZSB0cm9jYXIgZGUgYWxnb3JpdG1vLiIsICJDb21wYXJlbSBhcyBiYXJyYXM6IG8gcXVlIG11ZG91IG1haXMsIHRyb2NhciBvIGFsZ29yaXRtbyBvdSBqdW50YXIgYSBtYXLDqT8iXSwgIlAzIjogWyJiIiwgIlR1cnZhIHBlcmRpZGE6IGEgw6FndWEgZXN0YXZhIGJhcnJlbnRhIGUgbyBtb2RlbG8gZGlzc2UgcXVlIGVzdGF2YSBsaW1wYS4gQWxndcOpbSBwb2RlIGNvbmZpYXIgbmVzc2EgcmVzcG9zdGEuIiwgIlF1YWwgZG9zIGVycm9zIHBvZGUgbGV2YXIgYWxndcOpbSBhIGNvbmZpYXIgbnVtYSDDoWd1YSBxdWUgZXN0w6EgYmFycmVudGE/Il19").decode())
_CASTANHO, _CARMIM, _CINZENTO = "#8D6E63", "#A51C30", "#6B6B6B"


def _pc(x, casas=1):
    return f"{100 * x:.{casas}f}".replace(".", ",") + " %"


def _n(x):
    return f"{int(x):,d}".replace(",", " ")


def _falta(*nomes):
    g = globals()
    em_falta = [n for n in nomes if n not in g or g[n] is ...]
    if em_falta:
        print("Ainda não: falta criar " + ", ".join(em_falta) + ". Tirem o # do início das linhas, troquem os ... e corram a célula.")
        return True
    return False


def _treinado(m, n):
    return getattr(m, "n_features_in_", None) == n


def _boosting():
    return GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=42)


def _p1():
    if _falta("rio"):
        return
    if not isinstance(rio, pd.DataFrame) or "turva" not in rio:
        print('Ainda não. Pista: rio = pd.read_csv("cor_do_rio_e_mare.csv")')
        return
    print(f"Certo! A tabela tem {_n(len(rio))} linhas e {rio.shape[1]} colunas. Cada linha é uma manhã num troço do rio.")
    display(rio.head())


def _p2():
    if _falta("parte_turva"):
        return
    if abs(float(parte_turva) - rio["turva"].mean()) > 1e-9:
        print('Ainda não. Pista: parte_turva = rio["turva"].mean()')
        return
    print(f"Certo! A água estava turva em {_pc(parte_turva)} das manhãs.")
    parte = rio.groupby("mes")["turva"].mean()
    plt.figure(figsize=(9, 3.6))
    plt.bar(parte.index, 100 * parte.values, color=_CASTANHO)
    plt.xticks(range(1, 13), ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"])
    plt.ylabel("manhãs turvas (%)")
    plt.title("A água turva em cada mês")
    plt.show()


def _p3():
    if _falta("PISTAS", "ROTULO"):
        return
    if ROTULO != "turva":
        print('Ainda não: o rótulo é a coluna "turva".')
    elif "turva" in PISTAS or "ndti" in PISTAS:
        print("Ainda não: a resposta não pode ser uma pista. Seria dar ao modelo a solução do exame.")
    elif not isinstance(PISTAS, list) or sorted(PISTAS) != sorted(_SEM_MARE):
        print('Ainda não. Pista: PISTAS = ["caudal_m3s", "chuva_7d_mm", "mes"]')
    else:
        print('Certo! Três pistas (X) e um rótulo (y). A maré fica de fora, por agora.')


def _p4():
    if _falta("treino", "teste"):
        return
    if treino["ano"].max() == 2022 and teste["ano"].min() == 2023 and len(treino) + len(teste) == len(rio):
        print(f"Certo! {_n(len(treino))} manhãs para treinar (2017 a 2022) e {_n(len(teste))} para o exame (2023 a 2026).")
    else:
        print('Ainda não. Pista: treino = rio[rio["ano"] <= 2022] e teste = rio[rio["ano"] >= 2023]')


def _p5():
    if _falta("sempre_turva"):
        return
    if abs(float(sempre_turva) - teste[ROTULO].mean()) > 1e-9:
        print("Ainda não. Pista: sempre_turva = teste[ROTULO].mean()")
        return
    print(f"Certo! Quem disser sempre «turva», sem modelo nenhum, acerta {_pc(sempre_turva)}. É esta a nota a bater.")


def _p6():
    if _falta("arvore", "nota_arvore"):
        return
    certa = accuracy_score(teste[ROTULO], arvore.predict(teste[PISTAS]))
    if not _treinado(arvore, 3) or abs(float(nota_arvore) - certa) > 1e-9:
        print("Ainda não. Pista: arvore.fit(treino[PISTAS], treino[ROTULO]) e nota_arvore = accuracy_score(teste[ROTULO], arvore.predict(teste[PISTAS]))")
        return
    ganhou = "ganha a" if certa > sempre_turva else "não ganha a"
    print(f"Certo! A árvore acerta {_pc(certa)} no teste: {ganhou} «dizer sempre turva» ({_pc(sempre_turva)}).")


def _p7a():
    if _falta("floresta", "nota_floresta"):
        return
    certa = accuracy_score(teste[ROTULO], floresta.predict(teste[PISTAS]))
    if not _treinado(floresta, 3) or abs(float(nota_floresta) - certa) > 1e-9:
        print("Ainda não. Pista: floresta.fit(treino[PISTAS], treino[ROTULO]) e calculem a nota no teste, como no Passo 6.")
        return
    print(f"Certo! A floresta de 100 árvores acerta {_pc(certa)} no teste.")


def _p7b():
    if _falta("boosting", "nota_boosting"):
        return
    certa = accuracy_score(teste[ROTULO], boosting.predict(teste[PISTAS]))
    if not _treinado(boosting, 3) or abs(float(nota_boosting) - certa) > 1e-9:
        print("Ainda não. Pista: boosting.fit(treino[PISTAS], treino[ROTULO]) e calculem a nota no teste, como no Passo 6.")
        return
    print(f"Certo! O Gradient Boosting acerta {_pc(certa)} no teste.")
    nomes = ["Árvore", "Floresta", "Gradient Boosting"]
    notas = [nota_arvore, nota_floresta, nota_boosting]
    plt.figure(figsize=(9, 3.4))
    plt.barh(nomes[::-1], [100 * n for n in notas[::-1]], color=_CINZENTO)
    plt.axvline(100 * sempre_turva, color=_CARMIM, lw=2.5)
    plt.text(100 * sempre_turva + 1.5, 1, f"dizer sempre\n«turva»: {_pc(sempre_turva)}", color=_CARMIM, va="center", fontweight="bold")
    for i, n in enumerate(notas[::-1]):
        plt.text(100 * n - 1, i, _pc(n), va="center", ha="right", color="white", fontweight="bold")
    plt.xlim(0, 100)
    plt.xlabel("acerto no teste (%)")
    plt.title("Três algoritmos, sem a maré")
    plt.show()
    if max(notas) <= sempre_turva:
        print("Nenhum passa da linha vermelha: mudar de algoritmo não resolve. Falta uma pista.")
    else:
        print("Quase não passam da linha vermelha: mudar de algoritmo ajuda muito pouco. Falta uma pista.")


def ver_mare():
    if _falta("rio"):
        return
    faixa = pd.cut(rio["altura_da_mare_m"], [-5, -0.5, 0.5, 5], labels=["maré baixa", "maré média", "maré alta"])
    parte = rio.groupby(faixa, observed=False)["turva"].mean()
    plt.figure(figsize=(7, 3.8))
    plt.bar(parte.index.astype(str), 100 * parte.values, color=[_CASTANHO, "#BCAAA4", "#D7CCC8"])
    for i, v in enumerate(parte.values):
        plt.text(i, 100 * v + 2, _pc(v, 0), ha="center", fontsize=12)
    plt.ylim(0, 105)
    plt.ylabel("manhãs turvas (%)")
    plt.title("A água turva, conforme a maré à hora do satélite")
    plt.show()


def _p8():
    if _falta("PISTAS_COM_MARE", "modelo", "nota_com_mare"):
        return
    if sorted(PISTAS_COM_MARE) != sorted(_COM_MARE):
        print('Ainda não. Pista: PISTAS_COM_MARE = PISTAS + ["altura_da_mare_m"]')
        return
    certa = accuracy_score(teste[ROTULO], modelo.predict(teste[PISTAS_COM_MARE]))
    if not _treinado(modelo, 4) or abs(float(nota_com_mare) - certa) > 1e-9:
        print("Ainda não. Pista: modelo.fit(treino[PISTAS_COM_MARE], treino[ROTULO]) e calculem a nota com teste[PISTAS_COM_MARE].")
        return
    print(f"Certo! Com a maré, o Gradient Boosting acerta {_pc(certa)} no teste (sem a maré: {_pc(nota_boosting)}).")
    criar = {"Árvore": lambda: DecisionTreeClassifier(max_depth=3, random_state=42),
             "Floresta": lambda: RandomForestClassifier(n_estimators=100, random_state=42), "Gradient Boosting": _boosting}
    sem, com = [], []
    for fazer in criar.values():
        sem.append(100 * accuracy_score(teste[ROTULO], fazer().fit(treino[_SEM_MARE], treino[ROTULO]).predict(teste[_SEM_MARE])))
        com.append(100 * accuracy_score(teste[ROTULO], fazer().fit(treino[_COM_MARE], treino[ROTULO]).predict(teste[_COM_MARE])))
    x = np.arange(3)
    plt.figure(figsize=(9, 3.8))
    plt.bar(x - 0.2, sem, 0.38, color=_CINZENTO, label="sem a maré")
    plt.bar(x + 0.2, com, 0.38, color=_CARMIM, label="com a maré")
    for i in range(3):
        plt.text(i - 0.2, sem[i] - 3, f"{sem[i]:.1f}".replace(".", ","), ha="center", va="top", color="white", fontweight="bold")
        plt.text(i + 0.2, com[i] - 3, f"{com[i]:.1f}".replace(".", ","), ha="center", va="top", color="white", fontweight="bold")
    plt.axhline(100 * sempre_turva, color="#1A1A1A", lw=1.5, ls="--", label="dizer sempre «turva»")
    plt.xticks(x, list(criar))
    plt.ylim(0, 118)
    plt.yticks(range(0, 101, 20))
    plt.ylabel("acerto no teste (%)")
    plt.title("O que muda mais: o algoritmo ou a pista?")
    plt.legend(loc="upper center", ncol=3, fontsize=10)
    plt.show()


def _p9():
    if _falta("previsto"):
        return
    certo = modelo.predict(teste[PISTAS_COM_MARE])
    if not np.array_equal(np.asarray(previsto), certo):
        print("Ainda não. Pista: previsto = modelo.predict(teste[PISTAS_COM_MARE])")
        return
    v, p = teste[ROTULO].to_numpy(), np.asarray(previsto)
    r = [[int(((v == 1) & (p == 1)).sum()), int(((v == 1) & (p == 0)).sum())],
         [int(((v == 0) & (p == 1)).sum()), int(((v == 0) & (p == 0)).sum())]]
    nomes = [["turva certa", "turva perdida"], ["falso alarme", "limpa certa"]]
    cores = [["#DCEFE3", "#F6D5D5"], ["#F6D5D5", "#DCEFE3"]]
    fig, ax = plt.subplots(figsize=(6, 4.2))
    for i in range(2):
        for j in range(2):
            ax.add_patch(plt.Rectangle((j, 1 - i), 1, 1, color=cores[i][j], ec="white", lw=3))
            ax.text(j + 0.5, 1.5 - i, f"{nomes[i][j]}\n{_n(r[i][j])}", ha="center", va="center", fontsize=13)
    ax.set_xlim(0, 2)
    ax.set_ylim(0, 2)
    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(["o modelo disse TURVA", "o modelo disse LIMPA"])
    ax.set_yticks([1.5, 0.5])
    ax.set_yticklabels(["era TURVA", "era LIMPA"])
    ax.tick_params(length=0)
    for lado in ax.spines.values():
        lado.set_visible(False)
    ax.set_title("A matriz de confusão (verde = certo, vermelho = erro)")
    plt.show()
    print(f"Certo! Em {_n(len(v))} manhãs de teste: {_n(r[1][0])} falsos alarmes e {_n(r[0][1])} manhãs turvas perdidas.")


def _p10():
    if _falta("altura_da_mare", "probabilidade"):
        return
    manha_certa = pd.DataFrame({"caudal_m3s": [20], "chuva_7d_mm": [5], "mes": [6], "altura_da_mare_m": [altura_da_mare]})
    certa = modelo.predict_proba(manha_certa[PISTAS_COM_MARE])[0, 1]
    if abs(float(probabilidade) - certa) > 1e-9:
        print("Ainda não. Pista: probabilidade = modelo.predict_proba(manha[PISTAS_COM_MARE])[0, 1]")
        return
    print(f"Certo! Com a maré a {str(altura_da_mare).replace('.', ',')} m, a probabilidade de a água estar turva é {_pc(certa, 0)}.")
    print("Mudem a altura da maré na célula e corram outra vez.")


_PASSOS = {1: _p1, 2: _p2, 3: _p3, 4: _p4, 5: _p5, 6: _p6, "7a": _p7a, "7b": _p7b, 8: _p8, 9: _p9, 10: _p10}


def verificar(passo):
    """Diz se o passo está certo e mostra o resultado. Nunca dá erro vermelho."""
    try:
        _PASSOS[passo]()
    except Exception as erro:
        print("Ainda não: alguma coisa falta ou está errada (" + type(erro).__name__ + "). Confirmem esta célula e o passo anterior.")


def responder(pergunta, resposta):
    """Corrige uma pergunta de escolha: a resposta é a letra entre aspas, por exemplo "a"."""
    if resposta is ...:
        print('Falta responder: troquem os ... pela letra entre aspas, por exemplo "a".')
        return
    certa, explicacao, dica = _RESPOSTAS[pergunta]
    print(("Certo! " + explicacao) if str(resposta).strip().lower().startswith(certa) else ("Ainda não. " + dica))


if _pronto:
    plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
    print("PRONTO. Podem começar pelo Passo 1.")
else:
    print("AINDA NÃO ESTÁ PRONTO: sigam as instruções acima.")

## Passo 1 · Abrir os dados (5 min)

**O que vamos fazer agora:** abrir a tabela com os dados do rio e ver o que tem cada coluna.  
**Porquê:** Um modelo aprende com exemplos, e esta tabela são os nossos exemplos.

As colunas são as do quadro **«De onde vem cada informação»**, lá em cima, mais o dia (`data`, `ano`, `mes`) e o pedaço do rio (`troco`).

In [ ]:
# rio = pd.read_csv("cor_do_rio_e_mare.csv")

verificar(1)

## Passo 2 · Olhar para os dados (5 min)

**O que vamos fazer agora:** ver em que parte das manhãs a água estava turva, e em que meses.  
**Porquê:** Antes de treinar, olha-se sempre para os dados, para perceber o problema.

A média de uma coluna de 0 e 1 dá a parte das linhas em que ela vale 1: aqui, a parte das manhãs em que a água estava turva.

In [ ]:
# parte_turva = rio["..."].mean()

verificar(2)

## Passo 3 · Escolher as pistas e o rótulo (7 min)

**O que vamos fazer agora:** dizer ao computador com que informação vai adivinhar (as **pistas**, X) e qual é a resposta certa (o **rótulo**, y).  
**Porquê:** O modelo só aprende com o que lhe damos.

Vamos começar com o que parece lógico: o caudal, a chuva e o mês.

In [ ]:
# PISTAS = ["caudal_m3s", "chuva_7d_mm", "..."]
# ROTULO = "..."

verificar(3)

## Passo 4 · Separar o treino e o teste (8 min)

**O que vamos fazer agora:** dividir a tabela em duas: o **treino** (2017 a 2022) e o **teste** (2023 a 2026).  
**Porquê:** O modelo aprende com o treino e faz o exame com o teste, em manhãs que nunca viu.

**A regra de ouro:** nunca se avalia um modelo com os exemplos com que ele aprendeu. Queremos um modelo para o futuro: por
isso o treino é o **passado** e o teste é o **futuro**.

In [ ]:
# treino = rio[rio["ano"] <= ...]
# teste = rio[rio["ano"] >= ...]

verificar(4)

**Para pensar.** Porque é que não se avalia um modelo com os exemplos do treino?

a) porque demora mais tempo  
b) porque o modelo já os viu: a nota sai boa de mais  
c) porque o computador não deixa

In [ ]:
resposta = ...      # a letra entre aspas, por exemplo "a"
responder("P1", resposta)

## Passo 5 · Encontrar a nota a bater (3 min)

**O que vamos fazer agora:** calcular a nota de quem não usa modelo nenhum e diz sempre «turva».  
**Porquê:** Um modelo só vale a pena se fizer melhor do que isto.

Se disséssemos **sempre «turva»**, acertaríamos a parte das manhãs de teste que foram mesmo turvas.

In [ ]:
# sempre_turva = teste[...].mean()

verificar(5)

## Passo 6 · Treinar uma árvore de decisão (10 min)

**O que vamos fazer agora:** treinar o primeiro modelo, uma **árvore de decisão**, e dar-lhe uma nota.  
**Porquê:** É o modelo mais fácil de perceber: faz perguntas de sim ou não. Vamos ver se bate a nota do Passo 5.

Três linhas, sempre as mesmas: **criar** o modelo, **treinar** (`fit`) só com o treino, e **avaliar** (`predict`) com o teste.

In [ ]:
# arvore = DecisionTreeClassifier(max_depth=3, random_state=42)
# arvore.fit(treino[PISTAS], treino[...])
# nota_arvore = accuracy_score(teste[ROTULO], arvore.predict(teste[...]))

verificar(6)

## Passo 7 · Treinar uma floresta e um Gradient Boosting (10 min)

**O que vamos fazer agora:** treinar dois algoritmos mais fortes e comparar os três.  
**Porquê:** Se a árvore não chegou, talvez o problema seja o algoritmo.

Os dois algoritmos:

- **Floresta aleatória:** 100 árvores **lado a lado**, que votam.
- **Gradient Boosting:** 300 árvores **em fila**, cada uma corrige os erros das anteriores. É o algoritmo do projecto.

In [ ]:
# floresta = RandomForestClassifier(n_estimators=100, random_state=42)
# floresta.fit(treino[PISTAS], treino[ROTULO])
# nota_floresta = accuracy_score(teste[ROTULO], floresta.predict(teste[...]))

verificar("7a")

In [ ]:
# boosting = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=42)
# boosting.fit(treino[PISTAS], treino[ROTULO])
# nota_boosting = accuracy_score(teste[ROTULO], boosting.predict(teste[...]))

verificar("7b")

## Passo 8 · A pista que faltava: a maré (10 min)

**O que vamos fazer agora:** olhar para a maré e juntá-la às pistas.  
**Porquê:** Se trocar de algoritmo não resolveu, talvez falte informação. O rio chega ao mar, e o mar pode mudar a cor da água.

O mar sobe e desce duas vezes por dia. Corram a célula e vejam o gráfico.

In [ ]:
ver_mare()

Com a **maré baixa**, a água está quase sempre turva; com a **maré alta**, quase sempre mais limpa. Agora juntem a coluna da maré às pistas e treinem outra vez o Gradient Boosting.

In [ ]:
# PISTAS_COM_MARE = PISTAS + ["..."]
# modelo = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=42)
# modelo.fit(treino[PISTAS_COM_MARE], treino[ROTULO])
# nota_com_mare = accuracy_score(teste[ROTULO], modelo.predict(teste[...]))

verificar(8)

**Para pensar.** O que fez o acerto subir mais?

a) trocar a árvore por um algoritmo mais forte  
b) juntar a pista certa: a maré

In [ ]:
resposta = ...      # a letra entre aspas, por exemplo "a"
responder("P2", resposta)

## Passo 9 · Ver os erros do modelo (7 min)

**O que vamos fazer agora:** ver que erros faz o modelo.  
**Porquê:** A nota diz quanto ele errou, mas não como, e nem todos os erros são igualmente graves.

A **matriz de confusão** separa os dois tipos de erro:

- **Falso alarme:** estava limpa e o modelo disse turva.
- **Turva perdida:** estava turva e o modelo disse limpa.

In [ ]:
# previsto = modelo.predict(teste[...])

verificar(9)

**Para pensar.** Se alguém usar o modelo para decidir se vai buscar água ao rio, qual dos erros é mais perigoso?

a) o falso alarme  
b) a turva perdida

In [ ]:
resposta = ...      # a letra entre aspas, por exemplo "a"
responder("P3", resposta)

## Passo 10 · Prever uma manhã nova (5 min)

**O que vamos fazer agora:** usar o modelo numa manhã que ele nunca viu.  
**Porquê:** É para isto que se treina um modelo: responder a casos novos.

Uma manhã de Junho, com pouco caudal (20 m³/s) e pouca chuva (5 mm). O modelo não diz só «sim» ou «não»: dá uma
**probabilidade**. Experimentem a maré a **-1**, depois a **0** e depois a **0.7** (em Python, as décimas têm ponto).

In [ ]:
# altura_da_mare = ...
# manha = pd.DataFrame({"caudal_m3s": [20], "chuva_7d_mm": [5], "mes": [6], "altura_da_mare_m": [altura_da_mare]})
# probabilidade = modelo.predict_proba(manha[PISTAS_COM_MARE])[0, 1]

verificar(10)

## Passo 11 · O script inteiro (5 min)

**O que vamos fazer agora:** ver tudo o que fizeram num só script e corrê-lo.  
**Porquê:** Para perceberem que os passos de hoje são um programa curto, que serve de base para outros problemas.

Leiam-no de cima para baixo: já conhecem cada linha. Depois corram-no.

In [ ]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# 1. Abrir os dados
tabela = pd.read_csv("cor_do_rio_e_mare.csv")

# 2. Escolher as pistas e o rótulo
pistas = ["caudal_m3s", "chuva_7d_mm", "mes", "altura_da_mare_m"]
rotulo = "turva"

# 3. Separar no tempo: treino com o passado, teste com o futuro
parte_treino = tabela[tabela["ano"] <= 2022]
parte_teste = tabela[tabela["ano"] >= 2023]

# 4. Treinar
modelo_final = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=42)
modelo_final.fit(parte_treino[pistas], parte_treino[rotulo])

# 5. Avaliar com dados que o modelo nunca viu
nota_final = accuracy_score(parte_teste[rotulo], modelo_final.predict(parte_teste[pistas]))
print("Acerto no teste:", round(100 * nota_final, 1), "%")

## O que aprenderam

- O caminho do Machine Learning: **dados, pistas e rótulo, treino e teste, treinar, avaliar, prever**.
- Um modelo tem de bater uma nota simples, como «dizer sempre turva».
- **A pista certa vale mais do que o algoritmo:** sem a maré, nenhum modelo aprendeu nada útil.
- O acerto não chega: é preciso ver **que erros** o modelo faz.

**E no projecto?** O modelo verdadeiro é também um Gradient Boosting de 300 árvores, mas usa 20 pistas (o caudal, a chuva, a época do
ano, a maré e a última medição do satélite). Prevê a **cor** da água por volta das 9h30. **Não diz se a água é segura para beber.**